# 📘 Logistic Regression — Concepts Notebook
### Atlas Skilltech University | Instructor Reference

---

**Purpose:** This notebook explains every topic in Logistic Regression using small, hand-crafted sample data so concepts are crystal clear before moving to real datasets.

**Structure:**
1. What is Logistic Regression?
2. The Logistic (Sigmoid) Function
3. From Linear to Logistic — intuition + math
4. Maximum Likelihood Estimation
5. EDA + Model Training
6. Confusion Matrix & Accuracy
7. Precision, Recall, F1 Score
8. ROC Curve & AUC
9. Performance Evaluation Summary
10. Multiclass Classification

> 💡 Every hard keyword is explained in a markdown cell **before** the code.

---
## 🔧 Step 0 — Import Libraries

**Keywords to know:**
- **numpy** — library for numbers and math operations
- **pandas** — library for tables (DataFrames)
- **matplotlib** — library for drawing charts
- **scikit-learn (sklearn)** — Python's most popular machine learning library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully ✅')

---
## 📌 Section 1 — What is Logistic Regression?

**Keywords to know:**
- **Classification** — a type of machine learning where the goal is to predict a *category* (e.g., Pass/Fail, Yes/No, Spam/Not Spam)
- **Binary Classification** — when there are exactly **2 categories** (0 or 1)
- **Dependent variable (target)** — the column we want to predict (e.g., "Pass")
- **Independent variables (features)** — the columns we use to make the prediction (e.g., "Hours Studied")

Despite having "Regression" in the name, **Logistic Regression is used for Classification**, not regression.
It predicts the **probability** that something belongs to a category.

**Example:** Given how many hours a student studied, what is the probability they pass the exam?

In [ ]:
# Sample dataset — Student Exam Pass/Fail
data = {
    'Hours_Studied': [1, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 9],
    'Pass':          [0, 0, 0,   0, 0,   0, 1,   1, 1,   1, 1,   1, 1,   1, 1]
}

df = pd.DataFrame(data)
print(df)
print('\nShape of data:', df.shape)
print('\nTarget distribution (0=Fail, 1=Pass):')
print(df['Pass'].value_counts())

In [ ]:
# Visualise the data
plt.figure(figsize=(8, 4))
plt.scatter(df['Hours_Studied'], df['Pass'], color='steelblue', s=80, zorder=5)
plt.xlabel('Hours Studied')
plt.ylabel('Pass (1) / Fail (0)')
plt.title('Student Exam Outcome vs Hours Studied')
plt.yticks([0, 1], ['Fail (0)', 'Pass (1)'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📌 Section 2 — The Logistic (Sigmoid) Function

**Keywords to know:**
- **Sigmoid function** — an S-shaped curve that squishes any number into the range (0, 1). Perfect for probabilities!
- **e** — Euler's number ≈ 2.718 (a mathematical constant)
- **z** — the raw score (a number that can be anything from -∞ to +∞)

**Formula:**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

- When z → +∞, σ(z) → 1
- When z → -∞, σ(z) → 0
- When z = 0,   σ(z) = 0.5

So the sigmoid function converts any raw score into a **probability between 0 and 1**.

In [ ]:
# Define the sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Test a few values
test_values = [-10, -3, 0, 3, 10]
print('z value  |  sigmoid(z)')
print('-' * 25)
for z in test_values:
    print(f'  {z:>5}   |  {sigmoid(z):.4f}')

In [ ]:
# Plot the sigmoid curve
z = np.linspace(-10, 10, 300)

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid(z), color='darkorange', linewidth=2.5, label='Sigmoid σ(z)')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Threshold = 0.5')
plt.axvline(0, color='gray', linestyle='--', alpha=0.3)
plt.xlabel('z  (raw score)')
plt.ylabel('σ(z)  (probability)')
plt.title('The Sigmoid / Logistic Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📌 Section 3 — From Linear Regression to Logistic Regression

### Why can't we use Linear Regression for classification?

**Keywords to know:**
- **Linear Regression** — predicts a continuous number (e.g., house price)
- **Threshold** — a cutoff point; if probability > 0.5, predict class 1, else class 0
- **Log-Odds / Logit** — the natural log of the odds ratio

**Linear Regression equation:**
$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \ldots$$

Problem: $\hat{y}$ can be negative or greater than 1 — not valid for probabilities!

**Logistic Regression fix:** We pass the linear equation through the sigmoid:
$$P(y=1) = \sigma(w_0 + w_1 x_1) = \frac{1}{1 + e^{-(w_0 + w_1 x_1)}}$$

**Log-Odds (Logit) form:**
$$\ln\left(\frac{P}{1-P}\right) = w_0 + w_1 x_1$$

So logistic regression is actually a **linear model on the log-odds scale**.

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression

X = df[['Hours_Studied']]
y = df['Pass']

# Fit linear regression
lin_reg = LinearRegression()
lin_reg.fit(X, y)

# Fit logistic regression
log_reg = LogisticRegression()
log_reg.fit(X, y)

x_range = np.linspace(0, 10, 300).reshape(-1, 1)

plt.figure(figsize=(10, 5))

# Left: Linear Regression
plt.subplot(1, 2, 1)
plt.scatter(df['Hours_Studied'], df['Pass'], color='steelblue', s=80, zorder=5)
plt.plot(x_range, lin_reg.predict(x_range), color='red', linewidth=2)
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.axhline(1, color='gray', linestyle='--', alpha=0.5)
plt.title('❌ Linear Regression (bad for classification)')
plt.xlabel('Hours Studied')
plt.ylabel('Pass Probability')
plt.ylim(-0.3, 1.3)
plt.grid(True, alpha=0.3)

# Right: Logistic Regression
plt.subplot(1, 2, 2)
plt.scatter(df['Hours_Studied'], df['Pass'], color='steelblue', s=80, zorder=5)
plt.plot(x_range, log_reg.predict_proba(x_range)[:, 1], color='darkorange', linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.5')
plt.title('✅ Logistic Regression (correct for classification)')
plt.xlabel('Hours Studied')
plt.ylabel('Probability of Passing')
plt.ylim(-0.1, 1.1)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 📌 Section 4 — Maximum Likelihood Estimation (MLE)

**Keywords to know:**
- **Parameters (weights)** — the numbers w₀, w₁, w₂, ... that the model learns
- **Likelihood** — the probability that the model would produce the data we actually observed
- **Maximum Likelihood Estimation (MLE)** — find the parameter values that make the observed data **most probable**
- **Log-Likelihood** — we take the log of the likelihood (makes math easier, same optimal point)
- **Loss function (Log Loss / Binary Cross-Entropy)** — the quantity we minimise; it is the negative log-likelihood

**Log Loss formula:**
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

Where:
- $y_i$ = actual label (0 or 1)
- $\hat{p}_i$ = predicted probability
- $N$ = number of samples

Gradient descent (or similar optimisation) adjusts weights to **minimise** this loss.

In [ ]:
from sklearn.metrics import log_loss

# Predict probabilities
X = df[['Hours_Studied']]
y = df['Pass']

log_reg = LogisticRegression()
log_reg.fit(X, y)

probs = log_reg.predict_proba(X)[:, 1]

print('Learned weights (coefficients):', log_reg.coef_)
print('Intercept (bias / w0):', log_reg.intercept_)
print()
print('Log Loss (lower is better):', round(log_loss(y, probs), 4))

# Show predicted probabilities alongside actual labels
result_df = df.copy()
result_df['Predicted_Prob'] = probs.round(3)
result_df['Predicted_Class'] = (probs >= 0.5).astype(int)
print()
print(result_df)

---
## 📌 Section 5 — EDA + Model Training (Proper Pipeline)

**Keywords to know:**
- **EDA (Exploratory Data Analysis)** — exploring the data with charts and statistics before modelling
- **Train-Test Split** — split data: train the model on one part, test its performance on the other
- **Overfitting** — model memorises training data, fails on new data
- **random_state** — a seed number; ensures the same split every time you run the code
- **test_size=0.2** — 20% of data for testing, 80% for training

We'll use a slightly larger sample dataset here.

In [ ]:
# Larger sample dataset — 100 students with Hours Studied + Previous Score
np.random.seed(42)
n = 100

hours    = np.round(np.random.uniform(1, 10, n), 1)
prev     = np.round(np.random.uniform(30, 95, n), 1)
# Pass if hours > 5 OR prev > 65, with some noise
prob_raw = 0.5 * (hours / 10) + 0.5 * (prev / 100)
noise    = np.random.normal(0, 0.1, n)
y_true   = ((prob_raw + noise) > 0.5).astype(int)

students_df = pd.DataFrame({
    'Hours_Studied': hours,
    'Previous_Score': prev,
    'Pass': y_true
})

print('Dataset shape:', students_df.shape)
print()
students_df.head(10)

In [ ]:
# --- EDA ---
print('Basic Statistics:')
print(students_df.describe())
print()
print('Missing values:')
print(students_df.isnull().sum())
print()
print('Class balance (Pass column):')
print(students_df['Pass'].value_counts())

In [ ]:
# EDA Charts
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Class distribution
students_df['Pass'].value_counts().plot(kind='bar', ax=axes[0], color=['salmon', 'steelblue'], edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Pass (1) / Fail (0)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Fail (0)', 'Pass (1)'], rotation=0)

# 2. Hours studied by outcome
pass_students = students_df[students_df['Pass'] == 1]['Hours_Studied']
fail_students = students_df[students_df['Pass'] == 0]['Hours_Studied']
axes[1].hist(pass_students, alpha=0.7, color='steelblue', label='Pass', bins=10)
axes[1].hist(fail_students, alpha=0.7, color='salmon', label='Fail', bins=10)
axes[1].set_title('Hours Studied by Outcome')
axes[1].set_xlabel('Hours Studied')
axes[1].legend()

# 3. Scatter
colors = ['salmon' if p == 0 else 'steelblue' for p in students_df['Pass']]
axes[2].scatter(students_df['Hours_Studied'], students_df['Previous_Score'], c=colors, s=50, alpha=0.7)
axes[2].set_title('Hours vs Previous Score (colour = outcome)')
axes[2].set_xlabel('Hours Studied')
axes[2].set_ylabel('Previous Score')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Features (X) and target (y)
X = students_df[['Hours_Studied', 'Previous_Score']]
y = students_df['Pass']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples  : {len(X_train)}')
print(f'Testing  samples  : {len(X_test)}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scaling — keywords:
#   StandardScaler: transforms each feature so mean=0, std=1
#   We fit on training data only, then transform both train and test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train the model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print('Model trained ✅')
print('Coefficients:', model.coef_)
print('Intercept   :', model.intercept_)

---
## 📌 Section 6 — Confusion Matrix & Accuracy

**Keywords to know:**
- **Confusion Matrix** — a 2×2 table comparing actual vs predicted values
- **True Positive (TP)** — model said Pass, actual was Pass ✅
- **True Negative (TN)** — model said Fail, actual was Fail ✅
- **False Positive (FP)** — model said Pass, actual was Fail ❌ (Type I Error)
- **False Negative (FN)** — model said Fail, actual was Pass ❌ (Type II Error)
- **Accuracy** — % of predictions that were correct = (TP + TN) / Total

```
            Predicted Fail   Predicted Pass
Actual Fail      TN                FP
Actual Pass      FN                TP
```

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score
)

y_pred = model.predict(X_test_scaled)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2%}')

# Confusion matrix — the numbers
cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix (raw numbers):')
print(cm)

In [ ]:
# --- METHOD 1 (older style): matplotlib heatmap ---
import seaborn as sns

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fail', 'Pass'],
            yticklabels=['Fail', 'Pass'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — seaborn heatmap (older style)')
plt.tight_layout()
plt.show()

In [ ]:
# --- METHOD 2 (modern, recommended): ConfusionMatrixDisplay ---
# ConfusionMatrixDisplay is built into sklearn and handles everything automatically

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — ConfusionMatrixDisplay (modern style)')
plt.tight_layout()
plt.show()

---
## 📌 Section 7 — Precision, Recall, F1 Score

**Why not just use accuracy?**
If 95 out of 100 students pass, a model that always predicts "Pass" gets 95% accuracy — but it's useless!
We need better metrics.

**Keywords to know:**
- **Precision** — of all predictions of Pass, how many were actually Pass?
  $$\text{Precision} = \frac{TP}{TP + FP}$$
- **Recall (Sensitivity)** — of all actual Pass students, how many did we correctly catch?
  $$\text{Recall} = \frac{TP}{TP + FN}$$
- **F1 Score** — harmonic mean of Precision and Recall (balances both):
  $$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **Classification Report** — a table showing all these metrics per class
- **Support** — number of actual samples in each class

**Rule of thumb:**
- Use **Precision** when False Positives are costly (e.g., spam filter)
- Use **Recall** when False Negatives are costly (e.g., cancer diagnosis)
- Use **F1** when you need a balance

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

# Individual scores
print('Precision :', round(precision_score(y_test, y_pred), 4))
print('Recall    :', round(recall_score(y_test, y_pred), 4))
print('F1 Score  :', round(f1_score(y_test, y_pred), 4))
print()

# Full report
print('Full Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

---
## 📌 Section 8 — ROC Curve & AUC

**Keywords to know:**
- **Threshold** — the cutoff probability above which we predict class 1 (default is 0.5)
- **True Positive Rate (TPR / Sensitivity / Recall)** — fraction of actual positives correctly identified
  $$TPR = \frac{TP}{TP + FN}$$
- **False Positive Rate (FPR)** — fraction of actual negatives wrongly called positive
  $$FPR = \frac{FP}{FP + TN}$$
- **ROC Curve** — plots TPR vs FPR at every possible threshold
- **AUC (Area Under the Curve)** — a single number between 0 and 1; closer to 1 is better
  - AUC = 0.5 → random guessing
  - AUC = 1.0 → perfect model
- **RocCurveDisplay** — modern sklearn way to plot the ROC curve (recommended)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

y_proba = model.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print(f'AUC Score: {auc:.4f}')

In [ ]:
# --- METHOD 1 (older): manual plot using roc_curve ---
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve — Manual (older style)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- METHOD 2 (modern, recommended): RocCurveDisplay ---
disp = RocCurveDisplay.from_estimator(model, X_test_scaled, y_test)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.title('ROC Curve — RocCurveDisplay (modern style)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📌 Section 9 — Logistic Regression Performance Evaluation Summary

Let's put ALL evaluation metrics together in one clean summary.

**Keywords to know:**
- **Specificity (True Negative Rate)** — TN / (TN + FP); how well the model identifies negatives
- **Balanced Accuracy** — average of sensitivity and specificity; better for imbalanced datasets

In [ ]:
from sklearn.metrics import balanced_accuracy_score

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print('=' * 45)
print('     LOGISTIC REGRESSION PERFORMANCE SUMMARY')
print('=' * 45)
print(f'  True Positives  (TP) : {tp}')
print(f'  True Negatives  (TN) : {tn}')
print(f'  False Positives (FP) : {fp}')
print(f'  False Negatives (FN) : {fn}')
print('-' * 45)
print(f'  Accuracy         : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision        : {precision_score(y_test, y_pred):.4f}')
print(f'  Recall           : {recall_score(y_test, y_pred):.4f}')
print(f'  F1 Score         : {f1_score(y_test, y_pred):.4f}')
print(f'  AUC-ROC          : {roc_auc_score(y_test, y_proba):.4f}')
print('=' * 45)

---
## 📌 Section 10 — Multiclass Classification with Logistic Regression

**Keywords to know:**
- **Multiclass Classification** — predicting one of 3+ categories (e.g., Grade A/B/C)
- **One-vs-Rest (OvR)** — for each class, trains a binary classifier (this class vs all others)
- **Softmax** — generalises sigmoid to multiple classes; outputs a probability for each class
- **multi_class='multinomial'** — uses softmax for a true multi-class model
- **macro avg** — averages metric equally across all classes
- **weighted avg** — averages metric weighted by support (class size)

In [ ]:
# Create a 3-class sample dataset — Grade A, B, C
np.random.seed(0)
n = 150

hours2  = np.round(np.random.uniform(1, 10, n), 1)
score2  = np.round(np.random.uniform(30, 100, n), 1)

# Assign grades based on combined score
combined = 0.4 * hours2 + 0.006 * score2
grade = np.where(combined >= 0.75, 2,   # Grade A
        np.where(combined >= 0.5,  1,   # Grade B
                                   0))  # Grade C

multi_df = pd.DataFrame({
    'Hours_Studied': hours2,
    'Previous_Score': score2,
    'Grade': grade
})

print('Class distribution:')
print(multi_df['Grade'].value_counts().sort_index())
print(multi_df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

X_m = multi_df[['Hours_Studied', 'Previous_Score']]
y_m = multi_df['Grade']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42
)

scaler_m = StandardScaler()
X_train_m_sc = scaler_m.fit_transform(X_train_m)
X_test_m_sc  = scaler_m.transform(X_test_m)

# Multi-class logistic regression
multi_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', random_state=42)
multi_model.fit(X_train_m_sc, y_train_m)

y_pred_m = multi_model.predict(X_test_m_sc)

print('Multiclass Classification Report:')
print(classification_report(y_test_m, y_pred_m, target_names=['Grade C', 'Grade B', 'Grade A']))

In [ ]:
# Confusion matrix for multiclass
ConfusionMatrixDisplay.from_estimator(
    multi_model, X_test_m_sc, y_test_m,
    display_labels=['Grade C', 'Grade B', 'Grade A'],
    cmap='Blues'
)
plt.title('Multiclass Confusion Matrix')
plt.tight_layout()
plt.show()

---
## ✅ Summary — What We Covered

| Topic | Key Takeaway |
|---|---|
| Logistic Regression | Classification algorithm, NOT regression |
| Sigmoid Function | Squishes output to (0,1) for probability |
| Linear → Logistic | Linear model applied on log-odds scale |
| MLE / Log Loss | Optimisation method to find best weights |
| Train-Test Split | Evaluate on unseen data |
| Confusion Matrix | TP, TN, FP, FN breakdown |
| Precision / Recall / F1 | Better metrics than accuracy alone |
| ROC / AUC | Model's overall discriminative power |
| Multiclass | One-vs-Rest or Softmax for 3+ classes |

---
*Next: Apply all this on a real Kaggle dataset → see Notebook 2*